<a href="https://colab.research.google.com/github/johnisaiah09/FinanceResearch-TCS-/blob/main/Stock_Market_Research.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Imports and loading dataset
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras import Input
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay)
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.utils.class_weight import compute_class_weight
from google.colab import userdata
import requests
import time


API_KEY = userdata.get("ALPHA_VANTAGE_API_KEY")
data_raw = pd.read_csv("https://raw.githubusercontent.com/frontiertechinstitute/datasets/refs/heads/main/Major%20Tech%20Stocks%202019-2024%20Dataset/major-tech-stock-2019-2024.csv")
data = data_raw.copy()

In [ ]:
#EDA
print("Shape:", data.shape)
print("Columns:", data.columns)
print("Data types:", data.dtypes)
print("Date range:", data["Date"].min(), "to", data["Date"].max())
print("Companies:", data["Ticker"].unique())
print("Rows per company:", data["Ticker"].value_counts())

# Converting the Date column to datetime
data["Date"] = pd.to_datetime(data["Date"])

#Dropping Adj. Close
data = data.drop(columns=["Adj Close"])

# Checking data is sorted
data = data.sort_values(by=["Ticker", "Date"]).reset_index(drop=True)
data.head(10)

# Checking for invalid OHLC rows

invalid_rows = data[
    (data["High"] < data["Open"]) |
    (data["High"] < data["Close"]) |
    (data["High"] < data["Low"]) |
    (data["Low"] > data["Open"]) |
    (data["Low"] > data["Close"])
]

print(f"Number of invalid rows: {len(invalid_rows)}")

# Understanding numerical data
data.describe()

# Summary stats by Company
summary = data.groupby("Ticker").agg(
    {
        "Close": ["mean","std","min","max"],
        "Volume": ["mean","std","min","max"]
    }
)
summary

# Closing Price Visualization
companies = ["AAPL", "MSFT", "TSLA", "GOOGL", "AMZN"]

plt.figure(figsize=(12,6))
for company in companies:
  stock = data[data["Ticker"] == company]
  plt.plot(stock["Date"], stock["Close"], label=company)

plt.title("Closing Prices of 5 Tech Companies (2019-2024)")
plt.xlabel("Date")
plt.ylabel("Close Price ($)")
plt.legend()
plt.xticks(rotation=45)
plt.show()




In [ ]:
# Trading Volume Visualization
plt.figure(figsize=(15,6))

for ticker in data["Ticker"].unique():
  company = data[data["Ticker"] == ticker]
  plt.plot(
      company["Date"],
      company["Volume"],
      linewidth=1.8,
      label = ticker
  )

plt.title("Trading Volume of Five Tech Companies (2019-2023)", fontsize=15)
plt.xlabel("Date",fontsize=12)
plt.ylabel("Trading Volume (Millions of Shares)", fontsize=12)
plt.grid(True, alpha=0.3)
#Format y-axis in millions instead of sci-notation
plt.gca().yaxis.set_major_formatter(
    FuncFormatter(lambda x,pos: f"{x/1e6:.0f}M")
)

plt.legend(title="Company")
plt.tight_layout()
plt.show()

In [ ]:
from PIL.Image import linear_gradient
#Daily Returns for each company
data["Daily Return"] = (data.groupby("Ticker")["Close"].pct_change())

plt.figure(figsize=(15,6))

for ticker in data["Ticker"].unique():
  company = data[data["Ticker"] == ticker]

  plt.plot(
      company["Date"],
      company["Daily Return"],
      label=ticker,
      linewidth=1
  )

plt.title("Daily Returns of Five Tech Companies (2019-2023)", fontsize=15)
plt.xlabel("Date", fontsize=12)
plt.ylabel("Daily Return", fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


plt.figure(figsize=(10,6))

for ticker in data["Ticker"].unique():
  company = data[data["Ticker"] == ticker]

  plt.hist(
      company["Daily Return"],
      bins=50,
      alpha=0.5,
      label=ticker
  )

plt.title("Distribution of Daily Returns")
plt.xlabel("Daily Return")
plt.ylabel("Frequency")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
#Volatility
volatility = data.groupby("Ticker")["Daily Return"].std().sort_values(ascending=False)

plt.figure(figsize=(8,5))

volatility.plot(kind="bar")

plt.title("Volatility of Five Tech Companies (2019-2023)", fontsize=15)
plt.xlabel("Company", fontsize=12)
plt.ylabel("Volatiltiy/STD of Daily Returns", fontsize=12)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
#Correlation

corr = data[["Open", "High", "Low", "Close", "Volume"]].corr()
plt.figure(figsize=(6,5))

plt.imshow(corr,cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(label="Correlation")
plt.xticks(range(len(corr.columns)),corr.columns,rotation=45)
plt.yticks(range(len(corr.columns)),corr.columns)

for i in range(len(corr.columns)):
  for j in range(len(corr.columns)):
    plt.text(
        j,
        i,
        f"{corr.iloc[i,j]:.2f}",
        ha="center",
        va="center",
        color="black"
    )
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()



In [ ]:
#Data Preparation

#Get tomorrow's closing price for each company
data["Next_Close"] = data.groupby("Ticker")["Close"].shift(-1)

#Removing last trading day for each company
data = data.dropna(subset=["Next_Close"]).copy()

#Creating target
data["Target"] = (data["Next_Close"] > data["Close"]).astype(int)

#Getting rid of Next_Close
data.drop(columns=["Next_Close"], inplace=True)

#Checks (used once)
#print(data[["Date", "Ticker", "Close", "Target"]].head(10))
#print(data["Target"].value_counts())

In [ ]:
#Train/Test Split

#Training data: 2019-2022
train_data = data[data["Date"] < "2023-01-01"].copy()

#Testing data: 2023
test_data = data[data["Date"] >= "2023-01-01"].copy()

#Check 1
#print("Training period:")
#print(train_data["Date"].min(), "to", train_data["Date"].max())
#print()
#print("Testing period:")
#print(test_data["Date"].min(), "to", test_data["Date"].max())#

#Check2
#print("Training rows:", len(train_data))
#print("Testing rows:", len(test_data))

#Check3
#print(train_data["Ticker"].value_counts())
#print()
#print(test_data["Ticker"].value_counts())



In [ ]:
#Feature Scaling
features = ["Open","High","Low","Close","Volume"]
scaler = MinMaxScaler()
train_data[features] = scaler.fit_transform(train_data[features])
test_data[features] = scaler.transform(test_data[features])

#Checks
#print(train_data[features].describe())
#print(train_data[features].head())


In [ ]:
#Creating 15-Day Sequence
LOOKBACK = 15

features = ["Open","High", "Low", "Close", "Volume"]

#Function to create sequenes for one company
def create_sequences(data, features, target, lookback):

  X=[]
  y=[]

  for i in range(lookback, len(data)):

    #previous 30 trading days
    X.append(data[features].iloc[i-lookback:i].values)

    #Target for the current day
    y.append(data[target].iloc[i])

  return np.array(X), np.array(y)

#Empty list to store sequences
X_train_list = []
y_train_list = []

X_test_list = []
y_test_list = []

#Build sequences separately for each company
for ticker in train_data["Ticker"].unique():

  #Training data
  company_train = (
      train_data[train_data["Ticker"]==ticker].sort_values("Date").reset_index(drop=True)
  )
  X, y = create_sequences(company_train, features, "Target", LOOKBACK)
  X_train_list.append(X)
  y_train_list.append(y)

  #Testing data
  company_test = (
      test_data[test_data["Ticker"]==ticker].sort_values("Date").reset_index(drop=True)
  )
  X, y = create_sequences(company_test, features, "Target", LOOKBACK)
  X_test_list.append(X)
  y_test_list.append(y)

  #Combining companies into one dataset
  X_train = np.concatenate(X_train_list)
  y_train = np.concatenate(y_train_list)

  X_test = np.concatenate(X_test_list)
  y_test = np.concatenate(y_test_list)

  #Checks
#print("X_train shape:", X_train.shape)
#print("y_train shape:", y_train.shape)
#print("X_test shape:", X_test.shape)
#print("y_test shape:", y_test.shape)

#print("Example sequence shape:", X_train[0].shape)
#print("First target:", y_train[0])

In [ ]:
#Bulding model
model = Sequential([
    Input(shape=(LOOKBACK, len(features))),
    LSTM(64),
    Dropout(0.2),
    Dense(32, activation="relu"),
    Dense(1,activation="sigmoid")

])

# Compile the model
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

#Display
model.summary()

In [ ]:
#Training Baseline LSTM model

#Class weights
classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced",classes = classes, y=y_train)

class_weights = {0:weights[0], 1:weights[1]}
print(class_weights)



 #Early Stopping
early_stop = EarlyStopping(monitor="val_loss",patience=5,restore_best_weights=True)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    verbose=1,
    callbacks=[early_stop],
    class_weight=class_weights,
    shuffle=False
)

In [ ]:
#Evaluating on Test Set and getting predictions

#Eval on Test Set
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

#Predictions
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob >= 0.5).astype(int)
y_pred = y_pred.flatten()

print("First 10 Predictions:")
print(y_pred[:10])
print()
print("First 10 Actual Values:")
print(y_test[:10])

#Classifications/Confusion Matrix

#Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)
print("\nClassifaction Report:")
print(classification_report(y_test, y_pred))

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
#Testing with Alpha Vantage API
#url = "https://www.alphavantage.co/query"
#params = {"function": "NEWS_SENTIMENT", "tickers": "AAPL", "apikey": API_KEY, "time_from": "20190101T0000", "time_to": "20241231T2359", "limit": 1000, "sort": "EARLIEST"}
#response = requests.get(url, params=params)
#news_data = response.json()
#print(news_data.keys())

#Seeing what is given with feed
#article = news_data["feed"][0]
#article.keys()

In [ ]:
apple_news_df.head()

In [ ]:
print(apple_news_df["Date"].min())
print(apple_news_df["Date"].max())

In [2]:
#2019 Test request
#url = "https://www.alphavantage.co/query"
#params = {"function": "NEWS_SENTIMENT", "tickers": "AAPL", "time_from": "20190101T0000", "time_to": "20191231T2359", "sort": "EARLIEST", "limit":1000, "apikey": API_KEY}

response = requests.get(url, params=params)
aapl_2019_data = response.json()
#print(aapl_2019_data.keys())

#len(aapl_2019_data["feed"])

#Converting into DataFrame
articles = []

for article in aapl_2019_data["feed"]:
  articles.append({
      "Date": article["time_published"][:8],
      "Ticker": "AAPL",
      "Title": article["title"],
      "Sentiment Score": article["overall_sentiment_score"],
      "Summary": article["summary"],
      "SentimentLabel": article["overall_sentiment_label"],
      "Source": article["source"]
  })

aapl_2019_df = pd.DataFrame(articles)

#Converting to datetime format
aapl_2019_df["Date"] = pd.to_datetime(aapl_2019_df["Date"])
aapl_2019_df.head()

#Checking coverage
print(aapl_2019_df["Date"].min())
print(aapl_2019_df["Date"].max())


NameError: name 'aapl_2019_data' is not defined

In [ ]:
#Creating reusable function
def get_news_for_year(ticker,year,api_key):
  url = "https://www.alphavantage.co/query"
  params = {"function": "NEWS_SENTIMENT", "tickers": ticker, "time_from": f"{year}0101T0000", "time_to": f"{year}1231T2359", "sort": "EARLIEST", "limit":1000, "apikey": api_key}
  response = requests.get(url, params=params)
  news_data = response.json()

  #Prevent crashes
  if "feed" not in news_data:
    print("API Error:")
    print(news_data)
    return pd.DataFrame()


  articles = []

  for article in news_data["feed"]:
    articles.append({
        "Date": pd.to_datetime(article["time_published"][:8], format="%Y%m%d"),
        "Ticker": ticker,
        "Title": article["title"],
        "Sentiment Score": article["overall_sentiment_score"],
        "Summary": article["summary"],
        "SentimentLabel": article["overall_sentiment_label"],
        "Source": article["source"],
        "Year": year
    })

  time.sleep(12)
  return pd.DataFrame(articles)


In [ ]:
#Collect News Data for ALL Companies (2019-2023)
tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA"]
years = [2019, 2020, 2021, 2022, 2023]

apple_news = []

for year in years:
  print(f"Collecting AAPL {year}...")

  yearly_news = get_news_for_year("AAPL",year,API_KEY)

  apple_news.append(yearly_news)

apple_news_df = pd.concat(apple_news,ignore_index=True)

#Converting to datetime format
apple_news_df["Date"] = pd.to_datetime(apple_news_df["Date"], format="%Y%m%d")

#Preserve the raw news dataset
news_raw = apple_news_df.copy()


In [3]:
#Store company news datasets seperately
#news_datasets = {}

#Save AAPL dataset
#news_datasets["AAPL"] = apple_news_df

#Remaining Companies to collect
#remaining_tickers = ["MSFT", "TSLA", "GOOGL", "AMZN"]

#years = [2019,2020,2021,2022,2023]


#for ticker in remaining_tickers:
#print(f"\nStarting collection for {ticker}")

#  company_news = []

#for year in years:
 #   print(f"Collecting {ticker} {year}...")

  #  yearly_news = get_news_for_year(ticker,year,API_KEY)

  #  company_news.append(yearly_news)

  #company_df = pd.concat(company_news, ignore_index=True)

#  news_datasets[ticker] = company_df

#  print(f"{ticker} complete!")
#  print("Shape:", company_df.shape)



NameError: name 'apple_news_df' is not defined

In [ ]:
aapl_news_df.columns

In [22]:
test_response = requests.get(
    "https://www.alphavantage.co/query",
    params={
        "function": "NEWS_SENTIMENT",
        "tickers": "MSFT",
        "time_from": "20190101T0000",
        "time_to": "20191231T2359",
        "sort": "EARLIEST",
        "limit": 1000,
        "apikey": API_KEY
    }
)

test_data = test_response.json()

print(test_data.keys())
print(test_data)

dict_keys(['Information'])
{'Information': 'We have detected your API key as ZF83WZG8UO7X7WJH and our standard API rate limit is 25 requests per day. Please subscribe to any of the premium plans at https://www.alphavantage.co/premium/ to instantly remove all daily rate limits.'}


In [2]:
for ticker in remaining_tickers:
  for year in years:
    yearly_news = get_news_for_year(...)

NameError: name 'remaining_tickers' is not defined